# 03 Neural stream inventory and decoder manifest

This notebook bridges the trial catalog and the first decoding notebook.

## Purpose
1. Open every NWB file listed in the session index.
2. Inventory acquisition and processing objects across sessions.
3. Identify candidate neural streams for decoding.
4. Build a decoder manifest that maps each session to a likely neural source.
5. Save paper-style figures, tables, and metadata for the baseline decoder notebook.

> **Reference:** Issar et al. (2020) *J Neurophysiol* 123:1472–1485.  
> All figures in this notebook follow the aesthetic conventions of the reference paper:
> overlaid waveform traces, scatter-over-time with regression, dual-axis line plots,
> session-distribution histograms, and shaded mean ± SE curves.

## Inputs
- `/kaggle/working/tables_session_index/session_master_index.csv`
- `/kaggle/working/tables_session_index/decoder_trial_table.csv`
- Raw NWB files under: `/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N`

## Outputs
- `tables_stream_inventory/all_object_manifest.csv`
- `tables_stream_inventory/candidate_stream_manifest.csv`
- `tables_stream_inventory/session_decoder_stream_map.csv`
- `meta_stream_inventory/stream_inventory_report.json`
- `figures_stream_inventory/*.png / *.pdf`

In [ ]:
!pip install pynwb h5py --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 17.4 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 341.9/341.9 kB 23.0 MB/s eta 0:00:00


In [ ]:
import os
import re
import json
import math
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats

warnings.filterwarnings("ignore")
from pynwb import NWBHDF5IO

In [ ]:
# ─────────────────────────────────────────────
# PAPER-STYLE GLOBAL SETTINGS
# Mirrors Issar et al. 2020 (J Neurophysiol)
# ─────────────────────────────────────────────
plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 12,
    "axes.labelsize": 13,
    "axes.titlesize": 13,
    "axes.linewidth": 1.2,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "xtick.major.size": 6,
    "ytick.major.size": 6,
    "xtick.minor.size": 3,
    "ytick.minor.size": 3,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": 11,
    "legend.frameon": True,
    "legend.edgecolor": "0.4",
    "grid.linestyle": ":",
    "grid.linewidth": 0.7,
    "grid.alpha": 0.85,
})

# Paper palette (from Issar et al. Fig. 4–6)
C_BLACK   = "#1a1a1a"
C_MAROON  = "#8B2222"   # % waveforms removed line
C_BLUE    = "#2166AC"   # threshold crossings reference
C_GREEN   = "#1B7837"   # maximum decoding
C_GRAY    = "#888888"   # chance / control
C_ORANGE  = "#E08214"   # SpikingBandPower stream
C_RED_FILL = "#D6604D"  # NAS-helps shading

MARKER_PE  = "o"        # monkey Pe / primary subject
MARKER_WA  = "s"        # monkey Wa / secondary subject

def paper_axes(ax, top=True, right=True):
    ax.minorticks_on()
    ax.grid(True, which="major", linestyle=":", linewidth=0.8, alpha=0.85)
    ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.6)
    for spine in ax.spines.values():
        spine.set_linewidth(1.2)
    ax.tick_params(which="both", direction="in", top=top, right=right)

np.random.seed(42)

## Paths and notebook dependencies

This notebook expects the outputs from `02_session_index_and_trial_catalog.ipynb` to already exist.

In [ ]:
DATASET_DIR = Path("/kaggle/input/datasets/katakuricharlotte/dandi-dataset/001201/sub-Monkey-N")

IN_TABLE_DIR = Path("/kaggle/input/datasets/katakuricharlotte/02-session-index-results/tables_session_index")
SESSION_MASTER_CSV    = IN_TABLE_DIR / "session_master_index.csv"
DECODER_TRIAL_TABLE_CSV = IN_TABLE_DIR / "decoder_trial_table.csv"

OUT_FIG_DIR   = Path("/kaggle/working/figures_stream_inventory")
OUT_TABLE_DIR = Path("/kaggle/working/tables_stream_inventory")
OUT_META_DIR  = Path("/kaggle/working/meta_stream_inventory")

OUT_FIG_DIR.mkdir(parents=True, exist_ok=True)
OUT_TABLE_DIR.mkdir(parents=True, exist_ok=True)
OUT_META_DIR.mkdir(parents=True, exist_ok=True)

# Sample NWB for deep-dive waveform figures
SAMPLE_NWB = DATASET_DIR / "sub-Monkey-N_ses-20200127_ecephys.nwb"

print("DATASET_DIR         :", DATASET_DIR)
print("SESSION_MASTER_CSV  :", SESSION_MASTER_CSV)
print("DECODER_TRIAL_TABLE :", DECODER_TRIAL_TABLE_CSV)
print("SAMPLE_NWB          :", SAMPLE_NWB)

In [ ]:
assert DATASET_DIR.exists(),        f"Missing dataset directory: {DATASET_DIR}"
assert SESSION_MASTER_CSV.exists(), f"Missing prerequisite: {SESSION_MASTER_CSV}"
assert DECODER_TRIAL_TABLE_CSV.exists(), f"Missing prerequisite: {DECODER_TRIAL_TABLE_CSV}"
assert SAMPLE_NWB.exists(),         f"Missing sample NWB: {SAMPLE_NWB}"

session_master_df   = pd.read_csv(SESSION_MASTER_CSV)
decoder_trial_table = pd.read_csv(DECODER_TRIAL_TABLE_CSV)

if "session_date" in session_master_df.columns:
    session_master_df["session_date"] = pd.to_datetime(session_master_df["session_date"], errors="coerce")

if "days_since_first_session" not in session_master_df.columns:
    d0 = session_master_df["session_date"].min()
    session_master_df["days_since_first_session"] = (session_master_df["session_date"] - d0).dt.days

print("session_master_df shape :", session_master_df.shape)
print("decoder_trial_table shape:", decoder_trial_table.shape)
display(session_master_df.head(5))

## Helper functions

The following helpers safely inspect NWB objects without forcing large data reads into memory.

In [ ]:
def safe_attr(obj, attr, default=None):
    try:
        return getattr(obj, attr)
    except Exception:
        return default

def safe_len(x, default=np.nan):
    try:
        return len(x)
    except Exception:
        return default

def safe_shape(x):
    try:
        if hasattr(x, "shape") and x.shape is not None:
            return tuple(x.shape)
        return (len(x),)
    except Exception:
        return None

def to_jsonable(x):
    if isinstance(x, (str, int, float, bool)) or x is None:
        return x
    if isinstance(x, tuple):
        return list(x)
    if isinstance(x, list):
        return [to_jsonable(v) for v in x]
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    return str(x)

In [ ]:
def object_path(obj):
    parts = []
    seen  = set()
    cur   = obj
    while cur is not None and id(cur) not in seen:
        seen.add(id(cur))
        name = safe_attr(cur, "name", None)
        if name not in [None, "root"]:
            parts.append(str(name))
        cur = safe_attr(cur, "parent", None)
    return "/".join(reversed(parts))

def object_type(obj):
    ndt = safe_attr(obj, "neurodata_type", None)
    if ndt is not None:
        return str(ndt)
    return obj.__class__.__name__

def infer_container_group(path_text):
    txt = (path_text or "").lower()
    if "acquisition"  in txt: return "acquisition"
    if "processing"   in txt: return "processing"
    if "analysis"     in txt: return "analysis"
    if "stimulus"     in txt: return "stimulus"
    if "intervals"    in txt: return "intervals"
    return "other"

In [ ]:
def describe_nwb_object(obj):
    desc = {
        "name"         : safe_attr(obj, "name",         None),
        "neurodata_type": object_type(obj),
        "path"         : object_path(obj),
        "unit"         : safe_attr(obj, "unit",          None),
        "rate"         : safe_attr(obj, "rate",          None),
        "starting_time": safe_attr(obj, "starting_time", None),
        "description"  : safe_attr(obj, "description",  None),
        "comments"     : safe_attr(obj, "comments",      None),
    }
    data       = safe_attr(obj, "data",       None)
    timestamps = safe_attr(obj, "timestamps", None)
    desc["has_data"]        = data is not None
    desc["has_timestamps"]  = timestamps is not None
    desc["data_shape"]      = safe_shape(data)
    desc["timestamps_shape"]= safe_shape(timestamps)
    desc["container_group"] = infer_container_group(desc["path"])
    return desc

In [ ]:
def candidate_score(row):
    text = " ".join([
        str(row.get("name",         "")),
        str(row.get("neurodata_type","")),
        str(row.get("path",         "")),
        str(row.get("description",  "")),
        str(row.get("comments",     "")),
    ]).lower()

    score = 0
    positive_keywords = {
        "spike": 4, "waveform": 4, "threshold": 4,
        "electrical": 3, "ecephys": 3, "lfp": 3, "mua": 3, "multiunit": 3,
        "event": 2, "timeseries": 1,
    }
    negative_keywords = {
        "eye": -4, "gaze": -4, "pupil": -4,
        "fix": -3, "target": -3, "stim": -3, "reward": -3,
        "position": -2, "cursor": -2, "behavior": -2,
    }
    for k, v in positive_keywords.items():
        if k in text: score += v
    for k, v in negative_keywords.items():
        if k in text: score += v
    if row.get("has_data", False):          score += 1
    if pd.notna(row.get("rate", np.nan)):   score += 1
    if row.get("container_group") in ["acquisition", "processing"]: score += 1
    return score

In [ ]:
def inspect_single_nwb(nwb_path):
    rows = []
    with NWBHDF5IO(str(nwb_path), mode="r", load_namespaces=True) as io:
        nwb = io.read()
        base_meta = {
            "file_name"          : nwb_path.name,
            "file_path"          : str(nwb_path),
            "identifier"         : safe_attr(nwb, "identifier", None),
            "session_description": safe_attr(nwb, "session_description", None),
            "session_start_time" : str(safe_attr(nwb, "session_start_time", None)),
            "subject_id"         : safe_attr(safe_attr(nwb, "subject", None), "subject_id", None),
        }
        for obj in nwb.all_children():
            try:
                desc = describe_nwb_object(obj)
                if desc["name"] is None:
                    continue
                row = {**base_meta, **desc}
                row["candidate_score"] = candidate_score(row)
                rows.append(row)
            except Exception:
                continue
    return rows

## Build the all-object manifest

We scan every NWB file and create one row per discovered NWB object.
This mirrors the data inventory step described in Issar et al. (2020) §Materials and Methods,
where threshold-crossing streams and spiking-band power streams are identified as the
primary neural sources for decoding.

In [ ]:
nwb_files = sorted(DATASET_DIR.glob("*_ecephys.nwb"))
assert len(nwb_files) > 0, "No NWB files found."

all_rows = []
for i, nwb_path in enumerate(nwb_files, start=1):
    if i % 25 == 0 or i == 1 or i == len(nwb_files):
        print(f"Inspecting {i}/{len(nwb_files)}: {nwb_path.name}")
    rows = inspect_single_nwb(nwb_path)
    all_rows.extend(rows)

all_object_manifest = pd.DataFrame(all_rows)

# Merge session-level fields
merge_cols = [c for c in ["file_name", "session_index", "session_date",
                           "target_style_inferred", "days_since_first_session"]
              if c in session_master_df.columns]
if merge_cols:
    all_object_manifest = all_object_manifest.merge(
        session_master_df[merge_cols].drop_duplicates("file_name"),
        on="file_name", how="left"
    )

all_object_manifest["candidate_flag"] = all_object_manifest["candidate_score"] >= 4
all_object_manifest["data_shape"]      = all_object_manifest["data_shape"].apply(to_jsonable)
all_object_manifest["timestamps_shape"]= all_object_manifest["timestamps_shape"].apply(to_jsonable)

all_object_manifest.to_csv(OUT_TABLE_DIR / "all_object_manifest.csv", index=False)
print("all_object_manifest shape:", all_object_manifest.shape)
display(all_object_manifest.head(10))

In [ ]:
manifest_summary = (
    all_object_manifest
    .groupby(["neurodata_type", "container_group"], dropna=False)
    .size()
    .rename("n_objects")
    .reset_index()
    .sort_values("n_objects", ascending=False)
)
manifest_summary.to_csv(OUT_TABLE_DIR / "object_type_summary.csv", index=False)
display(manifest_summary.head(25))

In [ ]:
candidate_stream_manifest = (
    all_object_manifest
    .loc[all_object_manifest["candidate_flag"]]
    .sort_values(["file_name", "candidate_score"], ascending=[True, False])
    .reset_index(drop=True)
)
candidate_stream_manifest.to_csv(OUT_TABLE_DIR / "candidate_stream_manifest.csv", index=False)
print("candidate_stream_manifest shape:", candidate_stream_manifest.shape)
display(candidate_stream_manifest.head(20))

## Figure 1 — NWB object type inventory (horizontal bar chart)

This figure characterises what object types populate the dataset,
analogous to the data inventory step in Issar et al. (2020) §Materials and Methods.
The bar length encodes total object count across all 312 sessions.

In [ ]:
type_counts = (
    all_object_manifest
    .groupby("neurodata_type")
    .size()
    .rename("n_objects")
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(9, 5))

colors_bar = [C_MAROON if "Electrical" in t else C_BLUE if "TimeSeries" in t else C_GRAY
              for t in type_counts.index]

ax.barh(type_counts.index, type_counts.values,
        color=colors_bar, edgecolor="black", linewidth=0.7, height=0.65)

ax.set_xlabel("Total object count across all sessions")
ax.set_title("NWB neurodata-type inventory\n(Monkey N, 312 sessions)", pad=10)
paper_axes(ax)
ax.set_xlim(0, type_counts.max() * 1.15)

for val, y in zip(type_counts.values, range(len(type_counts))):
    ax.text(val + 30, y, str(val), va="center", fontsize=10)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=C_MAROON, edgecolor="black", label="ElectricalSeries (candidate)"),
    Patch(facecolor=C_BLUE,   edgecolor="black", label="TimeSeries"),
    Patch(facecolor=C_GRAY,   edgecolor="black", label="Other"),
]
ax.legend(handles=legend_elements, loc="lower right", frameon=True, fontsize=10)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig01_nwb_object_type_inventory.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig01_nwb_object_type_inventory.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 2 — Candidate score distribution (histogram)

Each NWB object receives a heuristic candidate score based on keyword matching
(spike, threshold, waveform → positive; eye, behavior, reward → negative).
Objects scoring ≥ 4 are flagged as candidate neural streams.

This mirrors the filtering step in the paper where threshold-crossing streams
and spiking-band power streams are isolated from all other NWB objects.

In [ ]:
scores = all_object_manifest["candidate_score"].dropna().values
threshold_score = 4

fig, ax = plt.subplots(figsize=(9, 5))

bins = np.arange(scores.min() - 0.5, scores.max() + 1.5, 1)

n_noise = (scores < threshold_score).sum()
n_cand  = (scores >= threshold_score).sum()

ax.hist(scores[scores < threshold_score], bins=bins,
        color=C_GRAY, edgecolor="black", linewidth=0.6, alpha=0.85, label=f"Non-candidate (n={n_noise})")
ax.hist(scores[scores >= threshold_score], bins=bins,
        color=C_MAROON, edgecolor="black", linewidth=0.6, alpha=0.9, label=f"Candidate (score ≥ {threshold_score}, n={n_cand})")

ax.axvline(threshold_score - 0.5, color="black", linestyle="--", linewidth=1.5, label=f"Threshold = {threshold_score}")
ax.set_xlabel("Candidate score")
ax.set_ylabel("Number of NWB objects")
ax.set_title("Distribution of candidate scores across all NWB objects", pad=10)
paper_axes(ax)
leg = ax.legend(loc="upper right", frameon=True, fancybox=False)
leg.get_frame().set_linewidth(0.9)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig02_candidate_score_distribution.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig02_candidate_score_distribution.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 3 — ThresholdCrossings and SpikingBandPower: T-samples over time

This scatter-over-time plot is directly analogous to **Fig. 5A–C** in Issar et al. (2020),
which tracks how waveform statistics change as a function of days since array implantation.
Here we plot the number of timepoints (T) per session for each candidate stream,
which serves as a proxy for recording duration stability across the 3-year span of this dataset.

In [ ]:
def extract_t_samples_from_shape(shape_val):
    """Parse serialised shape list/tuple and return first dimension (T-samples)."""
    if shape_val is None:
        return np.nan
    try:
        if isinstance(shape_val, list) and len(shape_val) >= 1:
            return int(shape_val[0])
        if isinstance(shape_val, tuple) and len(shape_val) >= 1:
            return int(shape_val[0])
        parsed = json.loads(str(shape_val))
        return int(parsed[0]) if isinstance(parsed, list) else np.nan
    except Exception:
        return np.nan

tc_df = (
    candidate_stream_manifest[candidate_stream_manifest["name"] == "ThresholdCrossings"]
    .copy()
    .sort_values("session_index")
    .reset_index(drop=True)
)
sbp_df = (
    candidate_stream_manifest[candidate_stream_manifest["name"] == "SpikingBandPower"]
    .copy()
    .sort_values("session_index")
    .reset_index(drop=True)
)

tc_df["T_samples"]  = tc_df["data_shape"].apply(extract_t_samples_from_shape)
sbp_df["T_samples"] = sbp_df["data_shape"].apply(extract_t_samples_from_shape)

print("ThresholdCrossings sessions:", len(tc_df))
print("SpikingBandPower sessions   :", len(sbp_df))
display(tc_df[["file_name", "session_index", "days_since_first_session", "T_samples", "data_shape"]].head())

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 9), sharex=True)

# ── Panel A: ThresholdCrossings T-samples vs days ──────────────────────────
ax = axes[0]
days_tc = tc_df["days_since_first_session"].values.astype(float)
T_tc    = tc_df["T_samples"].values.astype(float)
valid_mask = np.isfinite(days_tc) & np.isfinite(T_tc)

ax.scatter(days_tc[valid_mask], T_tc[valid_mask],
           s=28, color=C_MAROON, marker=MARKER_PE,
           edgecolor="black", linewidth=0.5, zorder=3,
           label="ThresholdCrossings")

if valid_mask.sum() >= 3:
    slope, intercept, r, p, _ = stats.linregress(days_tc[valid_mask], T_tc[valid_mask])
    x_line = np.linspace(days_tc[valid_mask].min(), days_tc[valid_mask].max(), 200)
    ax.plot(x_line, slope * x_line + intercept, color=C_BLACK, linewidth=1.8,
            label=f"ρ = {r:.2f}, p = {p:.2e}")

ax.set_ylabel("T-samples per session")
ax.set_title("A   ThresholdCrossings: recording length vs days since first session", pad=8, loc="left")
paper_axes(ax)
leg = ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=10)
leg.get_frame().set_linewidth(0.9)

# ── Panel B: SpikingBandPower T-samples vs days ────────────────────────────
ax = axes[1]
days_sbp = sbp_df["days_since_first_session"].values.astype(float)
T_sbp    = sbp_df["T_samples"].values.astype(float)
valid_sbp = np.isfinite(days_sbp) & np.isfinite(T_sbp)

ax.scatter(days_sbp[valid_sbp], T_sbp[valid_sbp],
           s=28, color=C_ORANGE, marker=MARKER_WA,
           edgecolor="black", linewidth=0.5, zorder=3,
           label="SpikingBandPower")

if valid_sbp.sum() >= 3:
    slope, intercept, r, p, _ = stats.linregress(days_sbp[valid_sbp], T_sbp[valid_sbp])
    x_line = np.linspace(days_sbp[valid_sbp].min(), days_sbp[valid_sbp].max(), 200)
    ax.plot(x_line, slope * x_line + intercept, color=C_BLACK, linewidth=1.8,
            label=f"ρ = {r:.2f}, p = {p:.2e}")

ax.set_xlabel("Days since first session")
ax.set_ylabel("T-samples per session")
ax.set_title("B   SpikingBandPower: recording length vs days since first session", pad=8, loc="left")
paper_axes(ax)
leg = ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=10)
leg.get_frame().set_linewidth(0.9)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig03_stream_Tsamples_vs_days.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig03_stream_Tsamples_vs_days.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 4 — Dual-axis: T-samples and target-style breakdown per session

**Panel A** shows the T-samples of ThresholdCrossings (black line, left axis)
alongside the number of trials per session (maroon line, right axis).
This dual-axis design mirrors **Fig. 4A–B** in Issar et al. (2020),
where % decoding accuracy (left axis) and % waveforms removed (right axis)
are overlaid against γ-threshold.

In [ ]:
# Compute trials per session from decoder_trial_table
if "file_name" in decoder_trial_table.columns:
    trials_per_session = (
        decoder_trial_table.groupby("file_name").size().rename("n_trials").reset_index()
    )
else:
    trials_per_session = pd.DataFrame(columns=["file_name", "n_trials"])

tc_merged = tc_df.merge(trials_per_session, on="file_name", how="left")
tc_merged = tc_merged.sort_values("session_index").reset_index(drop=True)

fig, ax1 = plt.subplots(figsize=(13, 5))

x_idx = tc_merged["session_index"].values
T_vals = tc_merged["T_samples"].values.astype(float)
n_trials_vals = tc_merged.get("n_trials", pd.Series(np.nan * len(tc_merged))).values.astype(float)

color_left  = C_BLACK
color_right = C_MAROON

ax1.plot(x_idx, T_vals,
         color=color_left, linewidth=2.2, marker=MARKER_PE,
         markersize=3.5, markerfacecolor="white", markeredgewidth=0.8,
         label="T-samples (ThresholdCrossings)")
ax1.set_xlabel("Session index")
ax1.set_ylabel("T-samples (ThresholdCrossings)", color=color_left)
ax1.tick_params(axis="y", labelcolor=color_left)
paper_axes(ax1)

ax2 = ax1.twinx()
ax2.plot(x_idx, n_trials_vals,
         color=color_right, linewidth=2.0, marker=MARKER_WA,
         markersize=3.5, markerfacecolor="white", markeredgewidth=0.8,
         label="Trials per session")
ax2.set_ylabel("Trials per session", color=color_right)
ax2.tick_params(axis="y", labelcolor=color_right, direction="in")
ax2.spines["right"].set_linewidth(1.2)

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2,
           loc="upper right", frameon=True, fancybox=False, fontsize=10)

ax1.set_title("Dual-axis: ThresholdCrossings T-samples and trial count per session", pad=10)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig04_dual_axis_Tsamples_trials.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig04_dual_axis_Tsamples_trials.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 5 — Sample waveforms from ThresholdCrossings (overlaid traces)

This is the most direct analogue of **Fig. 1A and Fig. 3A–D** in Issar et al. (2020),
which shows overlaid waveforms from individual channels colored by their spike/noise classification.

Here we read the actual `ThresholdCrossings` ElectricalSeries from the sample session
and plot:
- **Panel A**: raw overlaid traces for a single representative channel, coloured by
  estimated amplitude tertile (mimicking the spike-likelihood gradient in the paper)
- **Panel B**: the channel-average waveform ± 1 SD (shaded)
- **Panel C**: mean waveform per amplitude tertile across all 96 channels

> Note: the DANDI dataset provides per-bin threshold-crossing counts, not raw voltage
> waveform snippets (which are 52-sample cut-outs in the paper). We therefore plot
> the binned time-series cross-section for each channel, which is the equivalent
> observable in this dataset.

In [ ]:
N_WAVEFORMS_PLOT = 200   # max traces to overlay per panel
SAMPLE_CHANNEL   = 0     # channel index for per-channel panels

with NWBHDF5IO(str(SAMPLE_NWB), mode="r", load_namespaces=True) as io:
    nwb = io.read()

    # ── Find ThresholdCrossings ────────────────────────────────────────────
    tc_obj = None
    for obj in nwb.all_children():
        if safe_attr(obj, "name", "") == "ThresholdCrossings":
            tc_obj = obj
            break

    if tc_obj is not None:
        data_tc = tc_obj.data      # shape [T, 96]
        ts_tc   = tc_obj.timestamps

        # Read a manageable slice for plotting (first 3000 timepoints)
        T_SLICE = min(3000, data_tc.shape[0])
        tc_slice = data_tc[:T_SLICE, :]   # [T_SLICE, 96]
        ts_slice = ts_tc[:T_SLICE]

        print(f"ThresholdCrossings slice: {tc_slice.shape}, dtype: {tc_slice.dtype}")
    else:
        print("ThresholdCrossings object not found in sample NWB.")
        tc_slice = None
        ts_slice = None

In [ ]:
if tc_slice is not None:
    tc_arr  = np.array(tc_slice)   # [T, 96]
    ts_arr  = np.array(ts_slice)   # [T]
    n_ch    = tc_arr.shape[1]
    time_ms = (ts_arr - ts_arr[0]) * 1000.0   # convert to ms

    # Per-channel mean firing proxy (mean threshold-crossing count)
    ch_means = tc_arr.mean(axis=0)   # [96]
    sorted_ch = np.argsort(ch_means)[::-1]

    # Divide 96 channels into tertiles by mean activity
    t1 = sorted_ch[:32]   # high activity
    t2 = sorted_ch[32:64] # medium
    t3 = sorted_ch[64:]   # low activity

    COLORS_TERTILE = {
        "High (top 32 ch)":   "#D6604D",
        "Medium (mid 32 ch)": "#4393C3",
        "Low (bottom 32 ch)": "#888888",
    }
    TERTILE_DATA = {
        "High (top 32 ch)":   tc_arr[:, t1],
        "Medium (mid 32 ch)": tc_arr[:, t2],
        "Low (bottom 32 ch)": tc_arr[:, t3],
    }

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # ── Panel A: overlaid time-series per channel tertile (sample 200 ms window) ──
    ax = axes[0]
    WIN = min(500, len(time_ms))   # 500-sample window
    for label, color in COLORS_TERTILE.items():
        d = TERTILE_DATA[label][:WIN, :]
        for j in range(min(10, d.shape[1])):
            ax.plot(time_ms[:WIN], d[:, j],
                    color=color, alpha=0.35, linewidth=0.6)
        ax.plot(time_ms[:WIN], d[:, 0],
                color=color, linewidth=1.4, label=label)

    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Threshold crossings")
    ax.set_title("A   Overlaid traces by activity tertile", pad=8, loc="left")
    paper_axes(ax)
    leg = ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=9)
    leg.get_frame().set_linewidth(0.8)

    # ── Panel B: mean ± 1 SD per tertile ─────────────────────────────────
    ax = axes[1]
    for label, color in COLORS_TERTILE.items():
        d = TERTILE_DATA[label][:WIN, :]
        mu  = d.mean(axis=1)
        sd  = d.std(axis=1)
        ax.plot(time_ms[:WIN], mu, color=color, linewidth=2.0, label=label)
        ax.fill_between(time_ms[:WIN], mu - sd, mu + sd, color=color, alpha=0.2)

    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Mean ± 1 SD threshold crossings")
    ax.set_title("B   Mean ± 1 SD per tertile", pad=8, loc="left")
    paper_axes(ax)
    leg = ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=9)
    leg.get_frame().set_linewidth(0.8)

    # ── Panel C: per-channel mean activity, sorted (bar-code style) ───────
    ax = axes[2]
    sorted_means = np.sort(ch_means)[::-1]
    x_ch = np.arange(len(sorted_means))
    ax.bar(x_ch[:32], sorted_means[:32],  color="#D6604D", edgecolor="none", width=1.0, label="High")
    ax.bar(x_ch[32:64], sorted_means[32:64], color="#4393C3", edgecolor="none", width=1.0, label="Medium")
    ax.bar(x_ch[64:], sorted_means[64:],  color="#888888", edgecolor="none", width=1.0, label="Low")
    ax.set_xlabel("Channel rank (sorted by mean activity)")
    ax.set_ylabel("Mean threshold crossings")
    ax.set_title("C   Channel activity distribution (96 ch)", pad=8, loc="left")
    paper_axes(ax)
    leg = ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=9)
    leg.get_frame().set_linewidth(0.8)

    plt.suptitle(f"ThresholdCrossings: sample session {SAMPLE_NWB.name[:34]}",
                 fontsize=12, y=1.01)
    plt.tight_layout()
    plt.savefig(OUT_FIG_DIR / "fig05_sample_waveform_overlays.png", dpi=300, bbox_inches="tight")
    plt.savefig(OUT_FIG_DIR / "fig05_sample_waveform_overlays.pdf", dpi=300, bbox_inches="tight")
    plt.show()
else:
    print("Skipping Fig 5 — no ThresholdCrossings data available in sample NWB.")

## Figure 6 — SpikingBandPower vs ThresholdCrossings T-samples (joint scatter + marginals)

Directly modelled on **Fig. 6** in Issar et al. (2020), which uses a joint scatter plot
with marginal histograms to compare Δ% decoding accuracy for NAS vs manual spike-sorting.

Here we plot the joint distribution of T-samples for the two candidate streams
(ThresholdCrossings on one axis, SpikingBandPower on the other), with marginal histograms
on each axis. This confirms that both streams have matched temporal coverage
per session, a pre-condition for the fair comparison performed later in the decoding notebooks.

In [ ]:
tc_t  = tc_df.set_index("file_name")["T_samples"]
sbp_t = sbp_df.set_index("file_name")["T_samples"]
common_files = tc_t.index.intersection(sbp_t.index)
x_vals = tc_t.loc[common_files].values.astype(float)
y_vals = sbp_t.loc[common_files].values.astype(float)
valid  = np.isfinite(x_vals) & np.isfinite(y_vals)
x_vals, y_vals = x_vals[valid], y_vals[valid]

fig = plt.figure(figsize=(8, 8))
gs  = gridspec.GridSpec(4, 4, figure=fig)

ax_main   = fig.add_subplot(gs[1:, :3])
ax_top    = fig.add_subplot(gs[0, :3], sharex=ax_main)
ax_right  = fig.add_subplot(gs[1:, 3], sharey=ax_main)

# Joint scatter
ax_main.scatter(x_vals, y_vals,
                s=22, color=C_MAROON,
                edgecolor="black", linewidth=0.4, alpha=0.8, zorder=3)

# Identity line (y = x)
lim_min = min(x_vals.min(), y_vals.min()) * 0.97
lim_max = max(x_vals.max(), y_vals.max()) * 1.03
ax_main.plot([lim_min, lim_max], [lim_min, lim_max],
             color=C_GRAY, linewidth=1.5, linestyle="--", label="y = x")

# Regression line
slope, intercept, r, p, _ = stats.linregress(x_vals, y_vals)
x_reg = np.linspace(lim_min, lim_max, 200)
ax_main.plot(x_reg, slope * x_reg + intercept,
             color=C_BLACK, linewidth=2.0,
             label=f"ρ = {r:.3f}, p = {p:.2e}")

ax_main.set_xlabel("ThresholdCrossings T-samples")
ax_main.set_ylabel("SpikingBandPower T-samples")
paper_axes(ax_main)
leg = ax_main.legend(loc="upper left", frameon=True, fancybox=False, fontsize=10)
leg.get_frame().set_linewidth(0.9)

# Marginal top (ThresholdCrossings)
ax_top.hist(x_vals, bins=30, color=C_MAROON, edgecolor="black", linewidth=0.5, alpha=0.85)
ax_top.set_ylabel("Count")
ax_top.axvline(np.median(x_vals), color=C_BLACK, linewidth=1.5, linestyle="--")
paper_axes(ax_top)
plt.setp(ax_top.get_xticklabels(), visible=False)

# Marginal right (SpikingBandPower)
ax_right.hist(y_vals, bins=30, orientation="horizontal",
              color=C_ORANGE, edgecolor="black", linewidth=0.5, alpha=0.85)
ax_right.set_xlabel("Count")
ax_right.axhline(np.median(y_vals), color=C_BLACK, linewidth=1.5, linestyle="--")
paper_axes(ax_right)
plt.setp(ax_right.get_yticklabels(), visible=False)

fig.suptitle("Joint distribution of stream T-samples across sessions\n(ThresholdCrossings vs SpikingBandPower)",
             fontsize=12, y=1.01)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig06_joint_scatter_Tsamples.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig06_joint_scatter_Tsamples.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 7 — Session-distribution histograms (analogous to Fig. 4C–F)

In Issar et al. (2020) **Fig. 4C–D**, histograms show the distribution of
optimal γ-thresholds across sessions, and **Fig. 4E–F** show the distribution
of Δ% decoding accuracy. Here we apply the same layout to show the
session-level distribution of:
- **Panel C**: T-samples (ThresholdCrossings) — distribution across all sessions
- **Panel D**: T-samples (SpikingBandPower)
- **Panel E**: File size (MB) — proxy for recording richness
- **Panel F**: Days since first session — temporal coverage density

In [ ]:
file_sizes_mb = {p.name: p.stat().st_size / (1024**2) for p in nwb_files}
tc_merged["file_size_mb"]  = tc_merged["file_name"].map(file_sizes_mb)

fig, axes = plt.subplots(2, 2, figsize=(12, 9))

panels = [
    (axes[0, 0], tc_merged["T_samples"].dropna().values,
     "C   ThresholdCrossings T-samples per session", "T-samples", C_MAROON),
    (axes[0, 1], sbp_df["T_samples"].dropna().values,
     "D   SpikingBandPower T-samples per session", "T-samples", C_ORANGE),
    (axes[1, 0], tc_merged["file_size_mb"].dropna().values,
     "E   File size (MB) per session", "File size (MB)", C_BLUE),
    (axes[1, 1], session_master_df["days_since_first_session"].dropna().values,
     "F   Days since first session", "Days since first session", C_GRAY),
]

for ax, vals, title, xlabel, color in panels:
    ax.hist(vals, bins=30, color=color, edgecolor="black", linewidth=0.6, alpha=0.9)
    median_val = np.nanmedian(vals)
    ax.axvline(median_val, color=C_RED_FILL, linewidth=2.0, linestyle="--",
               label=f"Median = {median_val:.1f}")
    ax.set_xlabel(xlabel)
    ax.set_ylabel("Number of sessions")
    ax.set_title(title, pad=8, loc="left")
    paper_axes(ax)
    leg = ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=10)
    leg.get_frame().set_linewidth(0.9)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig07_session_distribution_histograms.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig07_session_distribution_histograms.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Figure 8 — Mean normalised T-samples: early vs late sessions (Fig. 5E analogue)

**Fig. 5E** in Issar et al. (2020) shows mean normalised decoding accuracy
as a function of γ-threshold, separately for early (0–50 days) and late (>50 days) sessions,
with ±1 SE shading.

Here we show the mean normalised T-samples (recording length) as a function of session rank,
stratified by early vs late recording epoch, with ±1 SE shading.
This reveals whether recording duration is consistent across the longitudinal span.

In [ ]:
EARLY_CUTOFF_DAYS = 180   # first ~6 months as "early"

tc_merged["epoch"] = tc_merged["days_since_first_session"].apply(
    lambda d: "Early (≤180 days)" if (pd.notna(d) and d <= EARLY_CUTOFF_DAYS) else "Late (>180 days)"
)

early_T = tc_merged.loc[tc_merged["epoch"] == "Early (≤180 days)", "T_samples"].dropna().values
late_T  = tc_merged.loc[tc_merged["epoch"] == "Late (>180 days)",  "T_samples"].dropna().values

overall_median = np.median(tc_merged["T_samples"].dropna().values)
early_norm = early_T / overall_median
late_norm  = late_T  / overall_median

# Rank within each epoch for x-axis
fig, ax = plt.subplots(figsize=(11, 5))

for norm_vals, label, color, marker in [
    (early_norm, "Early sessions (≤ 180 days)", C_BLUE,   "o"),
    (late_norm,  "Late sessions (> 180 days)",  C_MAROON, "s"),
]:
    ranks = np.arange(1, len(norm_vals) + 1)
    # Smooth with rolling mean for the shading
    window = max(1, len(norm_vals) // 8)
    from pandas import Series
    mu_series  = Series(norm_vals).rolling(window, center=True, min_periods=1).mean().values
    sd_series  = Series(norm_vals).rolling(window, center=True, min_periods=1).std().values
    se_series  = sd_series / np.sqrt(window)

    ax.plot(ranks, mu_series, color=color, linewidth=2.2,
            marker=marker, markersize=3.5, markeredgecolor="black",
            markeredgewidth=0.5, markevery=max(1, len(ranks)//20), label=label)
    ax.fill_between(ranks, mu_series - se_series, mu_series + se_series,
                    color=color, alpha=0.20)

ax.axhline(1.0, color=C_GRAY, linewidth=1.5, linestyle="--", label="Overall median (reference)")
ax.set_xlabel("Session rank within epoch")
ax.set_ylabel("Normalised T-samples (relative to overall median)")
ax.set_title("Mean normalised ThresholdCrossings T-samples: early vs late sessions\n(shading = ± 1 SE)", pad=10)
paper_axes(ax)
leg = ax.legend(loc="upper right", frameon=True, fancybox=False, fontsize=10)
leg.get_frame().set_linewidth(0.9)

plt.tight_layout()
plt.savefig(OUT_FIG_DIR / "fig08_normalised_Tsamples_early_late.png", dpi=300, bbox_inches="tight")
plt.savefig(OUT_FIG_DIR / "fig08_normalised_Tsamples_early_late.pdf", dpi=300, bbox_inches="tight")
plt.show()

## Build the session decoder stream map

For each session we assign the primary candidate stream (highest `candidate_score`).
In practice, every session has exactly two candidates: **ThresholdCrossings** (score 8)
and **SpikingBandPower** (score 4), matching the two signal types compared in the paper.

In [ ]:
session_decoder_stream_map = (
    candidate_stream_manifest
    .sort_values(["file_name", "candidate_score"], ascending=[True, False])
    .groupby("file_name")
    .first()
    .reset_index()
    [["file_name", "name", "neurodata_type", "candidate_score",
      "data_shape", "timestamps_shape", "rate", "unit",
      "session_index", "session_date", "target_style_inferred",
      "days_since_first_session"]]
    .rename(columns={"name": "primary_stream_name"})
)

session_decoder_stream_map.to_csv(OUT_TABLE_DIR / "session_decoder_stream_map.csv", index=False)
print("session_decoder_stream_map shape:", session_decoder_stream_map.shape)
display(session_decoder_stream_map.head(10))

## Stream inventory report (JSON)

In [ ]:
report = {
    "n_sessions_scanned"            : int(len(nwb_files)),
    "n_total_objects"               : int(len(all_object_manifest)),
    "n_candidate_objects"           : int(len(candidate_stream_manifest)),
    "candidate_stream_names_found"  : list(candidate_stream_manifest["name"].unique()),
    "sessions_with_ThresholdCrossings": int((candidate_stream_manifest["name"] == "ThresholdCrossings").sum() // 2 + 1),
    "sessions_with_SpikingBandPower"  : int((candidate_stream_manifest["name"] == "SpikingBandPower").sum() // 2 + 1),
    "median_T_samples_ThresholdCrossings": float(np.nanmedian(tc_df["T_samples"].values)),
    "median_T_samples_SpikingBandPower"  : float(np.nanmedian(sbp_df["T_samples"].values)),
    "primary_decoder_stream"       : "ThresholdCrossings",
    "n_channels_per_session"       : 96,
    "paper_reference"              : "Issar et al. 2020, J Neurophysiol 123:1472-1485",
}

with open(OUT_META_DIR / "stream_inventory_report.json", "w") as f:
    json.dump(report, f, indent=2, default=str)

print(json.dumps(report, indent=2, default=str))

In [ ]:
print("Generated figures:")
for p in sorted(OUT_FIG_DIR.glob("*")): print(" -", p.name)
print("\nGenerated tables:")
for p in sorted(OUT_TABLE_DIR.glob("*")): print(" -", p.name)
print("\nGenerated metadata:")
for p in sorted(OUT_META_DIR.glob("*")): print(" -", p.name)